In [6]:
print("something")
%pwd

something


'c:\\Users\\bhanu\\Desktop\\AI\\projects\\Medical_chatbot\\medical_chatbot\\research'

In [7]:
import os 
os.chdir("../")
%pwd

'c:\\Users\\bhanu\\Desktop\\AI\\projects\\Medical_chatbot\\medical_chatbot'

In [8]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [9]:
#Extract Data From the PDF File
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents

In [10]:
extracted_data=load_pdf_file(data='Data/')

In [11]:
print(type(extracted_data))
print(len(extracted_data))

<class 'list'>
637


In [12]:
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [13]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 5859


In [14]:
from langchain.embeddings import HuggingFaceEmbeddings

In [15]:
#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

embeddings = download_hugging_face_embeddings()

C:\Users\bhanu\AppData\Local\Temp\ipykernel_2972\220312833.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


In [16]:
query_result = embeddings.embed_query("Hello universe")
print("Length", len(query_result))

Length 384


In [17]:
len(query_result)

384

In [32]:
from dotenv import load_dotenv
import os
load_dotenv()   

True

In [25]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [20]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medicalbot"


if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)
    


In [21]:
# Embed each chunk and upsert the embeddings into your Pinecone index.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings, 
)

In [22]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [23]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [43]:
retriever2 = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":6})

In [28]:
from pprint import pprint
retrieved_docs = retriever.invoke("What is Antidepressant drugs?")
pprint(retrieved_docs)

[Document(id='517a4ad0-81e1-4b32-92d2-ba7a30adafa6', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 269.0, 'page_label': '270', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='CT 06105. (203) 232-4825.\nEpilepsy Foundation of America. 4351 Garden City Drive,\n#406, Landover, MD 20785. (800) 332-1000.\nNational Institute of Neurological Disorders and Stroke. P.O.\nBox 5801, Bethesda, MD 20824. (301) 496-5751.\nNancy Ross-Flanigan\nAntidepressant drugs\nDefinition\nAntidepressant drugs are medicines that relieve\nsymptoms of depressive disorders.\nPurpose\nDepressive disorders may be either unipolar (depres-\nsion alone) or bipolar (depression alternating with peri-'),
 Document(id='f8904ace-70cf-4ec8-90b0-a09f073ceed7', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 269.0, 

In [33]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o-mini")

In [39]:
response = chatModel.invoke("Hello! What is Elixir?")
print(response.content)

Elixir is a functional, concurrent programming language that runs on the Erlang Virtual Machine (BEAM). It was designed for building scalable and maintainable applications, particularly those that require high availability and low latency. Here are some key features of Elixir:

1. **Functional Programming**: Elixir encourages a functional programming paradigm, which means that it emphasizes the use of functions as first-class citizens and promotes immutability.

2. **Concurrency**: Leveraging the Erlang VM, Elixir inherits powerful features for concurrent programming. This allows developers to easily handle many tasks at once, making it an excellent choice for applications that need to manage multiple simultaneous connections, such as web servers and real-time systems.

3. **Fault Tolerance**: Elixir benefits from Erlang's "let it crash" philosophy, where failures are handled gracefully, allowing systems to recover quickly without significant downtime.

4. **Metaprogramming**: Elixir h

In [36]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [52]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer ONLY"
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [53]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever2, question_answer_chain)

In [54]:
# response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
# response = rag_chain.invoke({"input": "What is Elixir?"})
response = rag_chain.invoke({"input": "What is Ramayana?"})
print(response["answer"])


I don't know.


In [47]:
pprint(response)

{'answer': 'An elixir is a sweetened liquid that contains alcohol, water, and '
           'medicine. It is often used to deliver medication in a palatable '
           'form.',
 'context': [Document(id='1199a725-2cb3-4b74-93f0-13ee22463e49', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 283.0, 'page_label': '284', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='KEY TERMS\nElixir—A sweetened liquid that contains alcohol,\nwater, and medicine.\nFetus—A developing baby inside the womb.\nFungus—A unicellular to filamentous organism\nthat causes parasitic infections.\nOintment—A thick substance that contains medi-\ncine and is meant to be spread on the skin, or if an\nophthalmic ointment, in the eye.\nSystemic —A term used to describe a medicine\nthat has effects throughout the body, as opposed to\ntopical drugs that work on the skin. Most medi-'),
 